# 🚀 Approach 2: Balanced Sub-Sampling Bagging (10-Fold x 30 Bagging Rounds)

## 🧠 The Core Hypothesis:
In 96.2% vs 3.8% imbalanced datasets with 300+ features:
1. **Zero SMOTE Distortion:** In 300D space, SMOTE generates points in empty/negative regions. Balanced Bagging trains each sub-model on all 3,008 genuine positive samples + 3,008 genuine negative samples ($1:1$ balanced ratio).
2. **10-Fold Stratified CV (90% Train per Fold):** Guarantees each fold sees ~2,707 positive samples.
3. **30 Bagging Rounds per Fold (300 Total Sub-Models):** Covers the entire majority negative class space with zero noise.
4. **High-Resolution Threshold Scan (step=0.001):** Optimizes F1 with high precision.

**Hardware:** 100% CPU Friendly (Runs in ~2-3 minutes).


In [ ]:
import os, sys, time, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import f1_score, roc_auc_score

SEED = 42
N_FOLDS = 10
N_BAGGING_ROUNDS = 30
print('=' * 75)
print('  APPROACH 2: 10-FOLD BALANCED BAGGING ENSEMBLE (300 SUB-MODELS)')
print('=' * 75)


In [ ]:
CANDIDATE_DIRS = [
    '/kaggle/input/competitions/pstu-data-thon-2026-vol-1',
    '/kaggle/input/pstu-data-thon-2026-vol-1',
    'pstu-data-thon-2026-vol-1',
    '../input/competitions/pstu-data-thon-2026-vol-1',
    './Dataset',
    '.'
]
DATA_DIR = next((d for d in CANDIDATE_DIRS if os.path.exists(os.path.join(d, 'train.csv'))), None)
train_raw = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_raw  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

TARGET_COL = 'TARGET'
y = train_raw[TARGET_COL].copy()
test_ids = test_raw['id'].copy() if 'id' in test_raw.columns else pd.Series(range(len(test_raw)), name='id')
X_tr_raw = train_raw.drop(columns=[TARGET_COL])
X_te_raw = test_raw.drop(columns=['id']) if 'id' in test_raw.columns else test_raw.copy()

feat_cols = [c for c in X_tr_raw.columns if c.startswith('feat_')]
cat_cols  = X_tr_raw[feat_cols].select_dtypes(include=['object']).columns.tolist()
num_cols  = [c for c in feat_cols if c not in cat_cols]

X_num_tr = X_tr_raw[num_cols].apply(pd.to_numeric, errors='coerce').fillna(0).astype(np.float32)
X_num_te = X_te_raw[num_cols].apply(pd.to_numeric, errors='coerce').fillna(0).astype(np.float32)

# Outlier Clipping
p_low = np.percentile(X_num_tr, 1, axis=0)
p_high = np.percentile(X_num_tr, 99, axis=0)
X_num_tr = pd.DataFrame(np.clip(X_num_tr.values, p_low, p_high), columns=num_cols)
X_num_te = pd.DataFrame(np.clip(X_num_te.values, p_low, p_high), columns=num_cols)

# Simple Row Stats
row_tr = pd.DataFrame({'mean': X_num_tr.mean(axis=1), 'std': X_num_tr.std(axis=1), 'zero': (X_num_tr==0).sum(axis=1)})
row_te = pd.DataFrame({'mean': X_num_te.mean(axis=1), 'std': X_num_te.std(axis=1), 'zero': (X_num_te==0).sum(axis=1)})

scaler = RobustScaler()
X_tr_scaled = scaler.fit_transform(pd.concat([X_num_tr, row_tr], axis=1))
X_te_scaled = scaler.transform(pd.concat([X_num_te, row_te], axis=1))
print(f'Train matrix: {X_tr_scaled.shape} | Test matrix: {X_te_scaled.shape}')


## 2. Train 30 Balanced Sub-Sampling Classifiers across 10 Folds (300 Sub-Models)

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

oof_bagging = np.zeros(len(y), dtype=np.float32)
test_bagging = np.zeros(len(X_te_scaled), dtype=np.float32)

t_start = time.time()
print(f'Training {N_BAGGING_ROUNDS} Balanced Subsampled Models across {N_FOLDS} folds ({N_FOLDS*N_BAGGING_ROUNDS} models total)...')

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_tr_scaled, y)):
    X_tr_fold, y_tr_fold = X_tr_scaled[tr_idx], y.iloc[tr_idx].values
    X_va_fold, y_va_fold = X_tr_scaled[va_idx], y.iloc[va_idx].values
    
    pos_indices = np.where(y_tr_fold == 1)[0]
    neg_indices = np.where(y_tr_fold == 0)[0]
    n_pos = len(pos_indices)
    
    va_pred_sum = np.zeros(len(X_va_fold), dtype=np.float32)
    te_pred_sum = np.zeros(len(X_te_scaled), dtype=np.float32)
    
    for b in range(N_BAGGING_ROUNDS):
        rng = np.random.default_rng(SEED + fold * 100 + b)
        sampled_neg = rng.choice(neg_indices, size=n_pos, replace=False)
        sub_idx = np.concatenate([pos_indices, sampled_neg])
        rng.shuffle(sub_idx)
        
        X_sub, y_sub = X_tr_fold[sub_idx], y_tr_fold[sub_idx]
        
        clf = LogisticRegression(C=0.1, penalty='l2', max_iter=300, random_state=SEED+b)
        clf.fit(X_sub, y_sub)
        
        va_pred_sum += clf.predict_proba(X_va_fold)[:, 1] / N_BAGGING_ROUNDS
        te_pred_sum += clf.predict_proba(X_te_scaled)[:, 1] / (N_BAGGING_ROUNDS * N_FOLDS)
        
    oof_bagging[va_idx] = va_pred_sum
    test_bagging += te_pred_sum
    print(f'  Fold {fold+1:02d}/{N_FOLDS} Complete — OOF AUC: {roc_auc_score(y_va_fold, va_pred_sum):.4f}')

print(f'\nAll 300 Bagging passes completed in {time.time() - t_start:.1f}s.')


## 3. High-Resolution F1 Threshold Optimization & Export Submission

In [ ]:
thresholds = np.arange(0.10, 0.90, 0.001)
best_f1, best_t = 0.0, 0.5
for t in thresholds:
    b = (oof_bagging >= t).astype(int)
    if b.sum() == 0: continue
    f = f1_score(y.values, b)
    if f > best_f1: best_f1, best_t = f, t

print('=' * 75)
print(f'  10-FOLD BALANCED BAGGING OOF ROC-AUC: {roc_auc_score(y, oof_bagging):.5f}')
print(f'  🏆 OPTIMAL F1 THRESHOLD:              t = {best_t:.4f}')
print(f'  🏆 PEAK OOF F1-SCORE:                 F1 = {best_f1:.5f}')
print('=' * 75)

OUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'
binary_preds = (test_bagging >= best_t).astype(int)
sub = pd.DataFrame({'id': test_ids.values, 'TARGET': binary_preds})
sub.to_csv(os.path.join(OUT_DIR, 'submission.csv'), index=False)
print(f'Saved submission.csv with {int(binary_preds.sum()):,} predicted positives ({binary_preds.mean():.2%}).')
